In [1]:
import os
import glob
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [5]:
def compile_lfp_stats(
    pickle_dir="/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/lfp_spk_group_pickles"
):
    """
    Finds all sliding stats pickles in the directory and aggregates the pairwise 
    statistics into a master Pandas DataFrame for population-level analysis.
    """
    all_files = glob.glob(os.path.join(pickle_dir, "*_sliding_stats.pkl"))
    
    if not all_files:
        print(f"No pickle files found in {pickle_dir}")
        return pd.DataFrame()
    
    print(f"Found {len(all_files)} cell(s). Compiling data...")
    
    master_rows = []
    
    for file_path in all_files:
        with open(file_path, 'rb') as f:
            cell_data = pickle.load(f)
            
        cell_id = cell_data.get("cell_id", "Unknown")
        
        # Loop over every feature saved in this cell's dictionary
        for feature_name, stats in cell_data.items():
            if feature_name == "cell_id": 
                continue # Skip the ID key
                
            # Extract the pairwise stats
            pairwise_list = stats.get("pairwise_stats", [])
            
            for pair in pairwise_list:
                # Clean up group names for easier plotting (e.g., "Cluster: Low" -> "Low")
                g1 = pair['group_1'].split(': ')[-1] if ':' in pair['group_1'] else pair['group_1']
                g2 = pair['group_2'].split(': ')[-1] if ':' in pair['group_2'] else pair['group_2']
                
                # Standardize the comparison name (alphabetical so Low vs Mid is always the same)
                comp_name = f"{g1} vs {g2}"
                
                master_rows.append({
                    "cell_id": cell_id,
                    "feature": feature_name,
                    "window_start": pair["window_start"],
                    "window_end": pair["window_end"],
                    "group_1": g1,
                    "group_2": g2,
                    "comparison": comp_name,
                    "p_value": pair["p_value"],
                    "cohens_d": pair["cohens_d"]
                })
                
    df_master = pd.DataFrame(master_rows)
    return df_master



In [6]:
# Run the compiler
df_pop = compile_lfp_stats()

Found 7 cell(s). Compiling data...


In [7]:
df_pop

,cell_id,feature,window_start,window_end,group_1,group_2,comparison,p_value,cohens_d
0,c19,Aperiodic Exponent,0.553134,0.603134,low,high,low vs high,0.048897,0.069809
1,c19,Aperiodic Exponent,-0.174741,-0.124741,low,high,low vs high,0.035056,0.078845
2,c19,Aperiodic Exponent,0.000259,0.050259,low,high,low vs high,0.039892,0.077515
3,c19,Aperiodic Exponent,0.775259,0.825259,low,high,low vs high,0.032176,0.080178
4,c19,Aperiodic Offset,-0.174741,-0.124741,low,high,low vs high,0.045185,0.075000
...,...,...,...,...,...,...,...,...,...
141,c45,r-squared,-0.196882,-0.046882,low,mid,low vs mid,0.007828,0.088007
142,c45,r-squared,0.178118,0.378118,low,high,low vs high,0.012891,0.166814
143,c45,r-squared,0.178118,0.378118,mid,high,mid vs high,0.000808,0.223568
144,c45,r-squared,0.703118,0.878118,low,high,low vs high,0.001572,0.222453
